# Case Study Manual Validation

Sentence-level review for the top case study candidates: **Krone**, **ziehen**, **Blatt**.

**Goal:** Confirm that detected sense shifts are genuine language change, not WSD artifacts or genre drift.  
**Method:** Read 20–30 raw sentences per key transition decade; annotate whether the model's predicted sense matches your reading.

**Validation rubric per sentence:**
- ✓ = model prediction correct
- ✗ = model prediction wrong (note the actual sense)
- ? = ambiguous / unclear

See `diachronic_pilot_results.ipynb` Section 7 for the quantitative basis.

In [ ]:
import json
import pandas as pd
from pathlib import Path
from IPython.core.display import Markdown

BASE_DIR     = Path('../../data/processed/diachronic')
GERMANET_DIR = Path('../../data/processed/germanet')

preds = pd.read_csv(BASE_DIR / 'predictions/pilot_predictions_coarse.csv')

with open(GERMANET_DIR / 'sense_clusters_reviewed.json', encoding='utf-8') as f:
    clusters = json.load(f)

def show_clusters(lemma):
    print(f'Coarse senses for {lemma}:')
    for cid, cdata in clusters[lemma]['clusters'].items():
        print(f'  {cid}: {cdata["label"]}')

def show_proportions(lemma):
    lp = preds[preds['lemma'] == lemma]
    totals = lp.groupby('decade').size().rename('total')
    counts = lp.groupby(['decade', 'predicted_sense_id']).size().unstack(fill_value=0)
    props  = counts.div(totals, axis=0).round(3)
    print(f'\nProportion per decade:')
    display(pd.concat([counts, totals], axis=1))
    print(f'\nProportions:')
    display(props)

def validation_sentences(lemma, decades, n=25, seed=42):
    """Display sentences for given decades, grouped by decade and predicted sense."""
    lp = preds[preds['lemma'] == lemma]
    for dec in decades:
        dp = lp[lp['decade'] == dec]
        sample = dp.sample(n=min(n, len(dp)), random_state=seed).sort_values('predicted_sense_id')
        display(Markdown(f'---\n### {lemma} — {dec}s  (n={len(dp)} total, showing {len(sample)})' ))
        for _, row in sample.iterrows():
            label = clusters[lemma]['clusters'].get(row['predicted_sense_id'], {}).get('label', row['predicted_sense_id'])
            display(Markdown(
                f'**[{row["predicted_sense_id"]} | {label} | conf={row["confidence"]:.2f} | {int(row["year"])}]**  \n'
                f'{row["sentence"]}'
            ))

---
## 1. Krone

**Hypothesis:** `Krone_c0` (royal crown / Kopfbedeckung) declines as a proportion after the 1918 monarchy collapse.  
**Spearman ρ = −0.69** on `Krone_c0` proportion over decades.

| Decade | Krone_c0 % | n |
|--------|-----------|---|
| 1910s  | 80.5%     | 41 |
| 1920s  | 75.9%     | 29 |
| 1930s  | 70.6%     | 51 |
| 1950s  | 57.9%     | 19 |
| 1960s  | 50.0%     | 22 |
| 1990s  | 66.7%     | 18 |

**Key transition:** 1910s → 1960s (−30 pp). Focus on 1910s (peak) and 1950s–1960s (trough).

**What to look for:**
- Are 1910s `Krone_c0` sentences genuinely about royal/monarchical crowns? Or is the model over-predicting this sense?
- In 1950s–1960s, what is the model predicting instead? Are `Krone_c1` (currency) and `Krone_c2` (dental) sentences plausible?
- Any misclassifications that might artificially inflate or deflate the trend?

In [ ]:
show_clusters('Krone')
show_proportions('Krone')

In [ ]:
# Key validation decades: peak (1910s), steepest drop (1950s-1960s), recovery check (1990s)
validation_sentences('Krone', decades=[1910, 1950, 1960, 1990], n=25)

### Krone Validation Notes

*Fill in after reading sentences above.*

**1910s — observations:**
- 

**1950s — observations:**
- 

**1960s — observations:**
- 

**1990s — observations:**
- 

**Overall verdict:** [ ] Shift is genuine  [ ] Shift is artifact  [ ] Partial — explain:

**Estimated error rate (rough):** ___% misclassified

**Suitable as case study?** [ ] Yes  [ ] No  [ ] Yes with caveats

---
## 2. ziehen

**Hypothesis:** One sense of `ziehen` rises strongly over the century.  
**Spearman ρ = +0.842** — strongest monotonic trend among all candidates.

**What to look for:**
- Which sense is rising? Is the shift substantively interesting or trivial?
- Are predictions consistent and plausible, or is the model conflating senses?
- High-frequency verb — check whether genre drift (Belletristik → Wissenschaft) could explain the shift.

In [ ]:
show_clusters('ziehen')
show_proportions('ziehen')

In [ ]:
# Identify key transition decades from proportions above, then validate
# Default: early (1900s-1910s) vs late (1970s-1990s)
validation_sentences('ziehen', decades=[1900, 1910, 1970, 1980, 1990], n=20)

### ziehen Validation Notes

*Fill in after reading sentences above.*

**Early decades (1900s–1910s) — observations:**
- 

**Late decades (1970s–1990s) — observations:**
- 

**Rising sense — is it genuinely interesting?**
- 

**Genre drift concern:** [ ] Visible  [ ] Not visible  [ ] Unclear

**Overall verdict:** [ ] Shift is genuine  [ ] Shift is artifact  [ ] Partial — explain:

**Suitable as case study?** [ ] Yes  [ ] No  [ ] Yes with caveats

---
## 3. Blatt

**Hypothesis:** One sense of `Blatt` rises strongly over the century.  
**Spearman ρ = +0.810**.

**What to look for:**
- Which sense is rising? Newspaper/print (`Blatt` as newspaper) vs leaf vs sheet?
- Concrete noun — less genre-sensitive than verbs, more credible a priori.
- Any systematic misclassification between closely related senses?

In [ ]:
show_clusters('Blatt')
show_proportions('Blatt')

In [ ]:
validation_sentences('Blatt', decades=[1900, 1910, 1970, 1980, 1990], n=20)

### Blatt Validation Notes

*Fill in after reading sentences above.*

**Early decades (1900s–1910s) — observations:**
- 

**Late decades (1970s–1990s) — observations:**
- 

**Rising sense — is it genuinely interesting?**
- 

**Overall verdict:** [ ] Shift is genuine  [ ] Shift is artifact  [ ] Partial — explain:

**Suitable as case study?** [ ] Yes  [ ] No  [ ] Yes with caveats

---
## Summary

| Word | Shift | ρ | Verdict | Case study? |
|------|-------|---|---------|-------------|
| Krone | Krone_c0 declining | −0.69 | | |
| ziehen | TBD sense rising | +0.842 | | |
| Blatt | TBD sense rising | +0.810 | | |